# LPCMCI motorneuron baseline

Run this notebook independently of the motorneuron `oasis.ipynb`. Prepare the shared project environment once with `uv sync --frozen --all-extras`; the committed notebook extras preserve the required kernel tools, and selecting all extras prevents one baseline sync from removing the other. Results are descriptive because the recordings have no directed ground truth; raw PAG tensors remain primary.

<!-- reviewer-resume-contract -->
## Execution and resume contract

Each fluorescence-type–recording–representation fit is checkpointed. Inputs are the exact deduplicated records read by the c-GC/c-GC* motorneuron notebooks. Re-run the identical cell after interruption; configuration and input-file digests must match.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'empirical_baselines.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')


PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
RUNNER_ENV['PYTHONUNBUFFERED'] = '1'
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))
sys.path.insert(0, SOURCE_ROOT)

from calcium_transient_rising_flank.checkpointing import format_progress

NOTEBOOK_TAG = 'motorneurons/lpcmci.ipynb'
RUN_LPCMCI = True
FLUO_TYPES = 'dff,f_smooth'
RECORDINGS = 'F3T1,F3T2,F5T2'
REPRESENTATIONS = 'full,deconvolved,rise,fall,fall_residual'
OUTPUT_DIR = PACKAGE_ROOT / 'outputs/revision_campaign/motorneurons_lpcmci'
command = [
    RUNNER_PYTHON, 'examples/empirical_baselines.py',
    '--components', 'lpcmci', '--fluo-types', FLUO_TYPES,
    '--recordings', RECORDINGS,
    '--representations', REPRESENTATIONS,
    '--output-dir', str(OUTPUT_DIR), '--resume',
]
progress_path = OUTPUT_DIR / 'progress.json'
total_fits = len(FLUO_TYPES.split(',')) * len(RECORDINGS.split(',')) * len(REPRESENTATIONS.split(','))


def notebook_log(status: str, message: str) -> None:
    print(f'[{NOTEBOOK_TAG}] {status}: {message}', flush=True)


def show_resume_state(*, label: str, fallback_total: int) -> None:
    if progress_path.exists():
        saved = json.loads(progress_path.read_text())
        completed = int(saved.get('completed_unit_count') or 0)
        expected = int(saved.get('expected_unit_count') or fallback_total or 1)
        total = max(expected, 1)
        notebook_log('RESUMED', format_progress(min(completed, total), total, label=label))
        notebook_log(
            'RESUMED',
            f"status={saved.get('status')} active={saved.get('active_unit')} progress_file={progress_path}",
        )
    else:
        notebook_log('START', format_progress(0, max(fallback_total, 1), label=label))
        notebook_log('START', f'no saved progress at {progress_path}')


notebook_log('START', f'output={OUTPUT_DIR}')
notebook_log('START', f'planned work={total_fits} independent LPCMCI fits')
show_resume_state(label='Motorneuron LPCMCI fits', fallback_total=total_fits)
notebook_log('START', f'launching: {shlex.join(command)}')
if RUN_LPCMCI:
    subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
    show_resume_state(label='Motorneuron LPCMCI fits', fallback_total=total_fits)
    notebook_log('DONE', 'runner finished successfully')

summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
